# TrainAEClassifierHoldout optimized tutorial (all valid CV/test configs)

This notebook is intentionally thorough for optimized runs.

Included combinations:
- `cross_validation=False`, `cross_test=False`
- `cross_validation=True`, `cross_test=False`
- `cross_validation=True`, `cross_test=True`

`cross_test=True` is never used without cross-validation.
Each combination is run with `pools=False` and `pools=True`.

In [ ]:
from pathlib import Path

import pandas as pd

from bernn import TrainAEClassifierHoldout
from bernn.config.training_config import TrainingConfig

csv_path = Path('../data/benchmark/intensities.csv')
if not csv_path.exists():
    raise FileNotFoundError(f'Missing dataset: {csv_path.resolve()}')

df = pd.read_csv(csv_path)
X = df.iloc[:, 3:]
y = df.iloc[:, 1].to_numpy()
batches = df.iloc[:, 2].to_numpy()

split = max(8, int(0.8 * len(df)))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y[:split], y[split:]
batches_train, batches_test = batches[:split], batches[split:]

In [ ]:
# Important optimization-related parameters are explicit here.
bernn_config = TrainingConfig(
    optimize_hyperparams=True,
    n_trials=5,
    fixed_hyperparams={'scaler': 'standard'},
    n_repeats=2,
    n_layers=2,
    layer1=256,
    warmup=3,
    n_epochs=8,
    dloss='inverseTriplet',
    device='cpu',
    scaler='standard',
    bs=32,
)

# Set to False if you only want to inspect the configuration matrix.
run_all = False

In [ ]:
configurations = [
    {'cross_validation': False, 'cross_test': False},
    {'cross_validation': True, 'cross_test': False},
    {'cross_validation': True, 'cross_test': True},
]
pools_options = [False, True]

results = []

for pools in pools_options:
    for cfg in configurations:
        label = f"pools={pools}, cv={cfg['cross_validation']}, ct={cfg['cross_test']}"
        print(label)

        if not run_all:
            results.append({'config': label, 'status': 'skipped (run_all=False)'})
            continue

        trainer = TrainAEClassifierHoldout(
            config=bernn_config,
            pools=pools,
            log_metrics=True,
            keep_models=False,
        )

        _ = trainer.fit_predict(
            X_train,
            y_train,
            X_test=X_test,
            y_test=y_test,
            groups_train=batches_train,
            groups_test=batches_test,
            cross_validation=cfg['cross_validation'],
            cross_test=cfg['cross_test'],
        )

        preds = trainer.predict(X_test)
        results.append({'config': label, 'status': 'ok', 'pred_len': len(preds)})

pd.DataFrame(results)